# Alchemy GeomML + TDA — Kaggle Run (v33+)

Полный пайплайн обучения 7 моделей на Alchemy датасете через Kaggle GPU.

**ВАЖНО:** Запускай через **Save Version (Commit)**, не через интерактивную сессию!
Иначе output не сохранится.

**Датасет:** Alchemy v20191129 (202,579 молекул)

**Модели:** FCNN, SchNet, EGNN, EGNN+TDA, EGNN Vector, EGNN Vector+TDA, **EGNN Tensor** (часть B)

**GPU:** Kaggle T4 (16 ГБ) или P100 (16 ГБ) — бесплатно 30ч/неделю

**Оптимизация для быстрой сходимости:**
- `--epochs 9999` (EarlyStopping остановит раньше)
- `--patience 10` (вместо 15 — быстрее стопаем)
- `--es_mode or` (стоп при первой же метрике, которая перестала улучшаться)
- `--batch_size 1024` (T4 16GB это позволяет)

## 1. Подготовка окружения (2 минуты)

Клонируем репозиторий и проверяем версию.

In [ ]:
%cd /kaggle/working
import os
if not os.path.exists('alchemy-geom-tda'):
    !git clone https://github.com/ThomasMoore25/alchemy-geom-tda.git
else:
    %cd alchemy-geom-tda
    !git pull
%cd alchemy-geom-tda

# Подробный вывод версии и проверка фич
!git log --oneline -1
!echo ""
!echo "=== Проверка версии и фич ==="
!echo "PyGDataParallel: $(grep -c 'PyGDataParallel' src/train.py)"
!echo "DataListLoader:  $(grep -c 'DataListLoader' src/train.py)"
!echo "multi_gpu:       $(grep -c 'multi_gpu' src/train.py)"
!echo "num_workers:     $(grep -c 'num_workers' src/train.py)"
!echo "egnn_tensor:     $(grep -c 'egnn_tensor' src/train.py)"
!echo "physics.py:      $(ls -la src/physics.py 2>/dev/null && echo OK || echo MISSING)"
!echo "automl/run.py:   $(ls -la src/automl/run.py 2>/dev/null && echo OK || echo MISSING)"

## 2. Установка зависимостей (3 минуты)

Kaggle уже имеет torch + numpy + pandas. Ставим только недостающие — с подробным выводом.

In [ ]:
!pip install torch-geometric gudhi rdkit egnn-pytorch

# Подробная проверка окружения
import torch
import torch_geometric
import egnn_pytorch
import rdkit
import gudhi

print(f'\n=== Environment ===')
print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU count: {torch.cuda.device_count()}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ CUDA не доступна! Включи GPU: Settings → Accelerator → GPU T4 x2')
print(f'torch-geometric: {torch_geometric.__version__}')
print(f'rdkit: {rdkit.__version__}')
print(f'gudhi: {gudhi.__version__}')

import sys
sys.path.insert(0, '/kaggle/working/alchemy-geom-tda/src')
import src
print(f'\nalchemy-geom-tda version: {src.__version__}')

## 3. Скачивание датасета Alchemy (2 минуты)

136 МБ zip, ~600 МБ после распаковки. SHA256 проверка автоматически.

In [ ]:
# Полный вывод без | tail — видно прогресс скачивания и проверки
!python data/download_alchemy.py

## 4. Smoke-тест (5 минут, проверка что всё работает)

Обучаем EGNN на 100 молекулах, 2 эпохи. Должен быть виден убывающий loss.

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/alchemy-geom-tda/src')
os.makedirs('results/smoke', exist_ok=True)

sys.argv = ['train.py',
    '--model', 'egnn',
    '--target', 'all',
    '--epochs', '2',
    '--max_train', '100',
    '--max_val', '20',
    '--max_test', '20',
    '--batch_size', '32',
    '--device', 'cuda',
    '--output_dir', 'results/smoke',
    '--num_workers', '2',
]

from train import main as train_main
train_main()

## 5. Полное обучение — все 7 моделей (3-6 часов на T4 GPU)

Запуск через `run_all.py` — последовательно обучает все модели.

**Параметры оптимизированы для быстрой сходимости:**
- `--epochs 9999` — максимум (EarlyStopping остановит раньше)
- `--patience 10` — вместо 15, быстрее стопаем (для дедлайна)
- `--es_mode or` — стоп при первой метрике, которая перестала улучшаться
- `--batch_size 1024` — T4 16GB позволяет, ускоряет обучение
- `--multi_gpu` — если доступно 2+ GPU (Kaggle T4 x2)
- `--num_workers 4` — для ускорения DataLoader

In [ ]:
import os
import sys
sys.path.insert(0, '/kaggle/working/alchemy-geom-tda/src')

# ⚠️ ВСЕГДА запускай через Save Version (Commit), не через интерактив!
# Раскомментируй для запуска всех 7 моделей (3-6 часов на T4 GPU)

# sys.argv = ['run_all.py',
#     '--epochs', '9999',           # EarlyStopping остановит раньше
#     '--batch_size', '1024',        # T4 16GB позволяет
#     '--hidden_channels', '128',
#     '--num_layers', '4',
#     '--lr', '1e-3',
#     '--lr_patience', '5',
#     '--device', 'cuda',
#     '--patience', '10',            # быстрее стопаем (вместо 15)
#     '--es_mode', 'or',             # стоп при первой плохой метрике
#     '--models', 'all',             # или 'egnn,egnn_tda,egnn_vector,egnn_vector_tda,egnn_tensor'
#     '--num_workers', '4',
    '--multi_gpu',                 # раскомментируй если 2+ GPU
# ]

# from run_all import main as run_all_main
# run_all_main()

## 6. Альтернатива: только EGNN-семья (быстрее, ~2-4 часа)

FCNN и SchNet обучаются быстро — можно отдельно. EGNN-семья занимает больше всего времени.

In [ ]:
# Раскомментируй для запуска только EGNN-семьи (5 моделей)

# sys.argv = ['run_all.py',
#     '--epochs', '9999',
#     '--batch_size', '1024',
#     '--hidden_channels', '128',
#     '--num_layers', '4',
#     '--lr', '1e-3',
#     '--lr_patience', '5',
#     '--device', 'cuda',
#     '--patience', '10',
#     '--es_mode', 'or',
#     '--models', 'egnn,egnn_tda,egnn_vector,egnn_vector_tda,egnn_tensor',
#     '--num_workers', '4',
    '--multi_gpu',
# ]
# from run_all import main as run_all_main
# run_all_main()

## 7. Обучение только EGNN Tensor — часть B (1-2 часа)

**EGNN Tensor** (v33.0+, часть B) — физически корректная модель:
- Предсказывает **вектор** дипольного момента μ ∈ R³ (а не скаляр |μ|)
- Предсказывает **тензор** поляризуемости α ∈ R^(3×3) (а не скаляр tr(α)/3)
- Через частичные заряды атомов: q_i = MLP(h_atom)
- μ = Σᵢ qᵢ · (rᵢ − COM), α = Σᵢ qᵢ · (rᵢ − COM) ⊗ (rᵢ − COM)

**Чем отличается от обычного EGNN:**
- Обычный EGNN предсказывает только скаляр |μ| и скаляр α_iso
- EGNN Tensor предсказывает полный вектор μ и полный тензор α
- Это даёт больше информации: направление μ, анизотропию α
- E(3)-эквивариантность 1-го и 2-го ранга (l=1, l=2)

Запуск с `--predict_tensor_alpha` для полного тензора α.

In [ ]:
# Раскомментируй для запуска EGNN Tensor (часть B)

# sys.argv = ['train.py',
#     '--model', 'egnn_tensor',         # часть B: вектор μ + тензор α
#     '--target', 'all',
#     '--epochs', '9999',
#     '--batch_size', '1024',
#     '--hidden_channels', '128',
#     '--num_layers', '4',
#     '--lr', '1e-3',
#     '--lr_patience', '5',
#     '--device', 'cuda',
#     '--patience', '10',
#     '--es_mode', 'or',
#     '--predict_tensor_alpha',         # предсказывать полный тензор α
#     '--num_workers', '4',
#     '--output_dir', 'results/experiments/batch_size_1024',
# ]
# from train import main as train_main
# train_main()

## 8. Альтернатива: только одна модель (быстрая проверка, ~30-60 минут)

Обучить только EGNN на полном датасете с быстрым EarlyStopping.

In [ ]:
# Раскомментируй для запуска только EGNN (быстрая проверка)

# sys.argv = ['train.py',
#     '--model', 'egnn',
#     '--target', 'all',
#     '--epochs', '9999',
#     '--batch_size', '1024',
#     '--hidden_channels', '128',
#     '--num_layers', '4',
#     '--lr', '1e-3',
#     '--lr_patience', '5',
#     '--device', 'cuda',
#     '--patience', '10',
#     '--es_mode', 'or',
#     '--num_workers', '4',
#     '--output_dir', 'results/experiments/batch_size_1024',
# ]
# from train import main as train_main
# train_main()

## 9. Построение графиков обучения (1 минута)

Запускать после завершения обучения. Графики сохранятся в `results/figures/`.

In [ ]:
# После завершения обучения раскомментируй:

# sys.argv = ['plot.py',
#     '--input_dir', 'results/experiments/batch_size_1024',
#     '--save_dir', 'results/figures/batch_size_1024',
#     '--no-show',
# ]
# from plot import plot_main
# plot_main()

# Parity plots отдельно
# import glob
# from plot import plot_parity
# for csv in glob.glob('results/experiments/batch_size_1024/history_*.csv'):
#     model_name = csv.split('/')[-1].replace('history_', '').rsplit('_', 2)[0]
#     plot_parity(csv, save_path=f'results/figures/batch_size_1024/{model_name}_parity.png', show=False)

## 10. Robustness evaluation (30 минут на GPU)

Оценка устойчивости обученной модели к шуму в координатах.

In [ ]:
# Раскомментируй для robustness eval на EGNN

# sys.argv = ['eval_robustness.py',
#     '--model', 'egnn', '--target', 'all',
#     '--checkpoint', 'checkpoints/egnn_all_best.pt',
#     '--target_stats', 'results/experiments/batch_size_1024/target_stats_egnn_all.json',
#     '--noise_sigma', '0.0,0.05,0.10,0.15',
#     '--device', 'cuda',
# ]
# from eval_robustness import main as eval_main
# eval_main()

## 11. AutoML — автоматический выбор архитектуры (5 минут)

Программа максимум: извлечение геометрических prior-ов из TDA и рекомендация архитектуры.

Примечание: файл переименован из `select.py` в `run.py` (v33.2), чтобы избежать конфликта со стандартным модулем Python `select`.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/alchemy-geom-tda/src')
sys.path.insert(0, '/kaggle/working/alchemy-geom-tda/src/automl')

sys.argv = ['run.py',
    '--data_dir', 'data/alchemy',
    '--n_molecules', '100',
    '--threshold', '0.95',
    '--output_json', 'results/automl/recommendation.json',
    '--device', 'cpu',  # AutoML не требует GPU
]

# ВАЖНО: импорт из run, а не из select (конфликт со stdlib)
from run import main as automl_main
automl_main()

## 12. Просмотр отчёта AutoML

In [ ]:
import json
from pathlib import Path

report_path = 'results/automl/recommendation.json'
if Path(report_path).exists():
    with open(report_path) as f:
        report = json.load(f)
    print('=== AutoML Report ===')
    print(f'Timestamp: {report["timestamp"]}')
    print(f'\nPriors (n={report["priors"]["n_molecules_success"]} molecules):')
    for k, v in report['priors'].items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')
    print(f'\nRecommendation:')
    for k, v in report['recommendation'].items():
        print(f'  {k}: {v}')
else:
    print(f'Отчёт не найден: {report_path}')
    print('Запустите ячейку 11 сначала.')

## 13. Просмотр результатов обучения

In [ ]:
import pandas as pd
import glob

# Найти summary CSV
summary_csvs = sorted(glob.glob('results/experiments/batch_size_*/summary_*.csv'))
if summary_csvs:
    latest_summary = summary_csvs[-1]
    print(f'Latest summary: {latest_summary}')
    df = pd.read_csv(latest_summary)
    print(df.to_string(index=False))
else:
    print('Summary CSV не найден. Сначала запусти полное обучение (ячейка 5).')

# Последние 3 эпохи каждой модели
print('\n=== Last 3 epochs of each model ===')
for csv in sorted(glob.glob('results/experiments/batch_size_*/history_*.csv')):
    model_name = csv.split('/')[-1].replace('history_', '').rsplit('_', 2)[0]
    df = pd.read_csv(csv)
    print(f'\n{model_name} ({len(df)} epochs):')
    print(df.tail(3).to_string(index=False))

## 14. Тесты (краткий вывод)

Запускаем pytest с кратким выводом — только сводка в конце.

In [ ]:
# Краткий вывод: -q (quiet), --tb=line (одна строка на ошибку)
!python -m pytest -q --tb=line 2>&1 | tail -5

## 15. ⚠️ СОХРАНИТЬ ЧЕРЕЗ COMMIT

**НЕ запускай ячейку с обучением в interactive режиме!** Вместо этого:

1. Нажми **Save Version** в правом верхнем углу
2. Выбери **Save & Run All (Commit)**
3. Дай название версии (например: `v33 egnn+egnn_tensor bs1024`)
4. Нажми **Save**

Kaggle запустит ноутбук **в фоновом режиме** с нуля. После завершения:
- Output сохранится навсегда
- Появится в **Versions** tab
- Можно скачать через `kaggle kernels output` или через браузер

## 16. Что сохранится после Commit

В Output будут:
- `checkpoints/<model>_all_best.pt` — чекпойнт каждой модели
- `results/experiments/batch_size_1024/history_<model>_all_<ts>.csv` — история
- `results/experiments/batch_size_1024/summary_<ts>.csv` — сводка
- `results/experiments/batch_size_1024/args_<model>_all.json` — гиперпараметры
- `results/experiments/batch_size_1024/target_stats_<model>_all.json` — нормализация
- `results/figures/batch_size_1024/*.png` — графики
- `results/robustness/<model>_robustness.csv` — robustness
- `results/automl/recommendation.json` — отчёт AutoML
- `data/alchemy/processed/*.pt` — кэш TDA (для повторных запусков)

## 17. Частые проблемы

### OOM (CUDA out of memory)
Уменьши `--batch_size` до 512 или 256:
```python
'--batch_size', '512',
```

### DataParallel device mismatch
Раскомментируй `os.environ['CUDA_VISIBLE_DEVICES'] = '0'` в начале ноутбука.
Или используй `--device cuda:0` (v32+) для выбора конкретной GPU.

### Не нужны все 7 моделей
Сократи список:
```python
'--models', 'egnn,egnn_tda,egnn_tensor',  # только 3 модели
```

### Долгое обучение
Если Early Stopping занимает слишком много эпох:
- `--es_mode or` — стоп при первой плохой метрике (уже включено)
- `--patience 5` — ещё раньше останавливаться
- `--max_train 50000` — обучить на подвыборке (50k вместо 200k)

### TDA ухудшает метрики
Если `egnn_tda` работает хуже `egnn`:
- `--tda_mode film` — FiLM-модуляция вместо concat

### 12h лимит истечёт
При Commit Kaggle даёт 12h. Если не успел:
- Все файлы созданные до таймаута сохранятся
- Но если `torch.save()` не успел — чекпойнта не будет
- Можно запустить только EGNN + EGNN Tensor (без TDA-моделей — они медленнее)

## 18. Скачать результаты после Commit

```bash
# Через kaggle CLI
pip install kaggle
kaggle kernels output <username>/<notebook-slug> -p ~/alchemy_output

# Или через браузер
# Kaggle → notebook → Output tab → Download
```

## 19. Что делать с результатами

1. **Загрузить чекпойнты и CSV на GitHub** (через Git LFS — см. `.gitattributes`)
2. **Обновить `results/table.md`** финальными метриками
3. **Прогнать `eval_robustness.py`** на каждой модели для robustness таблицы
4. **Опубликовать v33 release** на GitHub с тегом `v33.2`